# Eval Overview - Combination Perturbation

Combination perturbation evaluation is a set of benchmarks that evaluate BioJEPA-AC's performance on samples where two or more genes are perturbed simultaneously. Biologically, these are important because genes interact: knocking out two genes in the same pathway can have a different effect than the sum of the individual knockouts. These non-additive effects, called genetic interactions, drive real phenomena like drug synergy and synthetic lethality.

This eval tests three things: (1) basic expression prediction quality on combo samples using the same metrics as the [expression prediction explainer](https://github.com/GPTomics/biojepa/blob/main/layer_explainers/explainer_eval_expr_prediction.ipynb), (2) whether the model does better than a naive additive baseline that just sums the individual perturbation effects, and (3) whether the model captures known genetic interaction types (synergistic, suppressive, etc.). Currently this eval only runs on the Norman dataset, which is the only dataset in our pipeline with combination perturbation samples and corresponding reference data from the [GEARS project](https://github.com/snap-stanford/GEARS). As you walk through the notebook you'll see that predicting combination effects is considerably harder than predicting single-perturbation effects, and the additive baseline is a surprisingly strong benchmark.

In [ ]:
import numpy as np
from collections import defaultdict
from scipy.stats import pearsonr

In [ ]:
SEED = 1337
np.random.seed(SEED)

## Data Prep

We'll start by preparing our data. For this evaluation, we need three categories of data: (1) the model's predictions on combination samples, (2) reference single-gene effects for each constituent gene, and (3) genetic interaction subtype labels. We also need a chain of mappings to connect our combo perturbation keys to their constituent gene pairs.

Since prior notebooks have walked through how sample-level deltas are computed, we'll stage per-combo aggregated deltas directly. We'll create 4 combo perturbations, each combining two genes, along with the single-gene reference data and genetic interaction labels needed for the additive baseline and subtype analysis.

In [ ]:
num_genes = 8
num_combos = 4
num_singles = 4
num_samples = 8

### Combination Perturbation Keys

In our other eval notebooks, we represent single perturbations with a 5-tuple: $\text{(seq id, targ id, modality id, mode id, cell type)}$. Combination perturbations can't use this format since there are multiple perturbations per sample. Instead, we build a *composite key* from each perturbation slot.

Each slot becomes either `('s', seq_idx, modality, mode)` if it has a sequence embedding or `('t', target_idx, modality, mode)` if it only has a target. We sort the slots so that the key for A+B and B+A is identical, then wrap them as a tuple alongside the cell type. This gives us a unique, order-independent key for each combination.

We'll create 4 combo perturbations, each combining two DNA CRISPRi knockdowns. Each combo has its own guide RNA construct, so the seq_idx values (10, 12, 14, 16) are unique per combo. We also need `first_seq_idx`, which maps each composite key to the seq_idx from its first sorted slot. This value is what connects a composite key to the gene pair lookup in the next step.

In [ ]:
combo_keys = [
    ((('t', 0, 0, 0), ('t', 1, 0, 0)), 0),  # GENE_1 + GENE_2, CRISPRi, cell type 0
    ((('t', 0, 0, 0), ('t', 2, 0, 0)), 0),  # GENE_1 + GENE_3, CRISPRi, cell type 0
    ((('t', 1, 0, 0), ('t', 3, 0, 0)), 0),  # GENE_2 + GENE_4, CRISPRi, cell type 0
    ((('t', 2, 0, 0), ('t', 3, 0, 0)), 0),  # GENE_3 + GENE_4, CRISPRi, cell type 0
]

first_seq_idx = {
    combo_keys[0]: 10,  # GENE_1+GENE_2 combo guide
    combo_keys[1]: 12,  # GENE_1+GENE_3 combo guide
    combo_keys[2]: 14,  # GENE_2+GENE_4 combo guide
    combo_keys[3]: 16,  # GENE_3+GENE_4 combo guide
}

combo_keys, first_seq_idx

### Single Gene Reference Deltas

To build the additive baseline, we need to know what each gene does when perturbed alone. During data prep, we compute mean expression deltas from all single-perturbation samples for each gene and store them in `norman_single_gene_deltas.npz`. Each row is one gene's mean expression delta across all measured genes.

We'll stage 4 single genes with distinct expression signatures. GENE_1 primarily affects genes 0-1, GENE_2 affects genes 2-3, GENE_3 affects genes 4-5, and GENE_4 affects genes 6-7. Each gene also has small spillover effects on other genes to keep the data realistic.

In [ ]:
single_gene_names = ['GENE_1', 'GENE_2', 'GENE_3', 'GENE_4']
single_deltas = np.array([
    [ 1.0,  0.5, -0.1,  0.0,  0.0,  0.0,  0.0, -0.1],  # GENE_1: strong on genes 0-1
    [-0.1,  0.0,  1.0,  0.5,  0.0,  0.0,  0.0,  0.0],   # GENE_2: strong on genes 2-3
    [ 0.0,  0.0,  0.0, -0.1,  1.0,  0.5,  0.0,  0.0],   # GENE_3: strong on genes 4-5
    [ 0.0, -0.1,  0.0,  0.0,  0.0,  0.0,  1.0,  0.5],   # GENE_4: strong on genes 6-7
])
single_deltas.shape, single_deltas

### Combo-to-Genes Mapping

Now we need to connect our composite keys to their constituent gene pairs. This requires a chain of lookups: composite key -> `first_seq_idx` -> `combo_mapping` -> gene pair names. The `combo_mapping` comes from `norman_combo_mapping.json` created during data prep, which maps a first-slot seq_idx to the pair of gene names that combo perturbs.

We'll stage the mapping and then build the final `combo_to_genes` dict by walking through each combo key, looking up the first_seq_idx, and then looking up the gene pair. We print each step so you can trace the chain.

In [ ]:
combo_mapping = {
    '10': ['GENE_1', 'GENE_2'],
    '12': ['GENE_1', 'GENE_3'],
    '14': ['GENE_2', 'GENE_4'],
    '16': ['GENE_3', 'GENE_4'],
}

combo_to_genes = {}
for key in combo_keys:
    sid = first_seq_idx.get(key)
    genes = combo_mapping.get(str(sid)) if sid is not None else None
    combo_to_genes[key] = tuple(genes) if genes else None
    print(f'Key {combo_keys.index(key)}: first_seq_idx={sid} -> combo_mapping["{sid}"] -> {combo_to_genes[key]}')

combo_to_genes

### Per-Combo Expression Deltas

These are the mean predicted and real expression deltas across all samples of each combo perturbation. In our production eval, they come from the multi-pert `RunningMeans` in test inference. Since we have a few different notebooks showing how sample-level deltas are computed, we'll stage them directly as per-perturbation means.

We've staged the data to show four different interaction patterns that you'll see emerge when we compare against the additive baseline:
- **GENE_1+GENE_2**: synergistic, the real effect is amplified well beyond the sum of singles
- **GENE_1+GENE_3**: alleviating, the genes partially cancel each other out
- **GENE_2+GENE_4**: roughly additive, the real effect is close to the sum
- **GENE_3+GENE_4**: suppressive, GENE_4 dominates and GENE_3's effect is nearly eliminated

In [ ]:
mean_real_deltas = {
    combo_keys[0]: np.array([1.5, 0.8, 1.5, 0.8, 0.1, 0.1, -0.1, -0.2]),      # GENE_1+GENE_2 synergistic
    combo_keys[1]: np.array([0.3, 0.1, -0.05, 0.0, 0.3, 0.15, 0.0, -0.05]),    # GENE_1+GENE_3 alleviating
    combo_keys[2]: np.array([-0.1, -0.05, 0.95, 0.45, 0.0, 0.0, 0.95, 0.45]),  # GENE_2+GENE_4 additive
    combo_keys[3]: np.array([0.05, -0.1, 0.0, 0.0, 0.15, 0.1, 1.1, 0.6]),      # GENE_3+GENE_4 suppressive
}

mean_pred_deltas = {
    combo_keys[0]: np.array([1.3, 0.7, 1.3, 0.7, 0.05, 0.05, -0.05, -0.15]),   # captures synergy
    combo_keys[1]: np.array([1.2, 0.6, -0.15, -0.1, 1.2, 0.6, 0.0, -0.1]),     # over-predicts, misses alleviating
    combo_keys[2]: np.array([-0.05, -0.05, 0.9, 0.45, 0.0, 0.0, 0.9, 0.45]),   # close to additive
    combo_keys[3]: np.array([0.0, -0.1, 0.0, -0.1, 1.2, 0.6, 0.9, 0.4]),       # over-predicts GENE_3, misses suppression
}

for i, key in enumerate(combo_keys):
    genes = combo_to_genes[key]
    print(f'Combo {i} ({genes[0]}+{genes[1]}):')
    print(f'  real: {mean_real_deltas[key]}')
    print(f'  pred: {mean_pred_deltas[key]}')

### Sample-Level Metrics

For the expression prediction sub-eval, we also need per-sample MSE and Pearson correlations. These are computed during test inference for every sample regardless of whether it's single or multi-pert. We stage 8 values (2 samples per combo), with the synergistic combo samples showing lower error and higher correlation than the alleviating and suppressive ones.

In [ ]:
sample_mses = np.array([0.015, 0.012, 0.25, 0.28, 0.001, 0.002, 0.17, 0.19])
sample_correlations = np.array([0.85, 0.88, 0.3, 0.25, 0.95, 0.92, 0.4, 0.35])
sample_mses.shape, sample_correlations.shape

### Genetic Interaction Subtypes

The Norman dataset comes with genetic interaction labels from the original paper, classified by the type of interaction observed: *synergistic* (combo effect exceeds the sum of singles), *alleviating* (combo effect is weaker, genes compensate), *suppressive* (one gene dominates), among others. These labels come from the [GEARS project](https://github.com/snap-stanford/GEARS) and are stored in `norman_gi_subtypes.json`.

We'll label 3 of our 4 combos to show how the per-subtype breakdown works. The fourth (GENE_2+GENE_4) will be unlabeled to show that not every combo has a known interaction type. We also build `name_to_subtype` with both orderings of each gene pair so the lookup works regardless of order. In the production code, gene names are single words without underscores (e.g. AHR, KLF1), so `combo_name.split('_')` gives exactly 2 parts and the reversal is automatic. Since our staged gene names contain underscores, we'll reverse using the known gene pairs from `combo_to_genes` directly.

In [ ]:
gi_subtypes = {
    'GENE_1_GENE_2': 'Synergistic',
    'GENE_1_GENE_3': 'Alleviating',
    'GENE_3_GENE_4': 'Suppressive',
}

name_to_subtype = {}
for combo_name, subtype in gi_subtypes.items():
    name_to_subtype[combo_name] = subtype
    print(f'{combo_name} -> {subtype}')

for key in combo_keys:
    genes = combo_to_genes[key]
    fwd = f'{genes[0]}_{genes[1]}'
    rev = f'{genes[1]}_{genes[0]}'
    if fwd in name_to_subtype and rev not in name_to_subtype:
        name_to_subtype[rev] = name_to_subtype[fwd]
        print(f'{rev} -> {name_to_subtype[fwd]} (reversed)')

name_to_subtype

## Expression Prediction on Combos

The first thing we evaluate is basic expression prediction quality on the combo subset. This uses the same metrics we cover in the [expression prediction explainer](https://github.com/GPTomics/biojepa/blob/main/layer_explainers/explainer_eval_expr_prediction.ipynb): MSE and Pearson correlation on the expression delta. The question here is whether the model can predict combo effects at all, before we ask whether it does better than the additive baseline.

We'll compute a few key metrics per combo to establish the baseline quality level. You'll see that the roughly-additive combo (GENE_2+GENE_4) has the best prediction quality, while the combos with strong interaction effects (alleviating and suppressive) are harder for the model.

In [ ]:
per_combo_mse = []
per_combo_pearson = []

for key in combo_keys:
    genes = combo_to_genes[key]
    pred = mean_pred_deltas[key]
    real = mean_real_deltas[key]

    mse = float(np.mean((pred - real) ** 2))
    per_combo_mse.append(mse)
    per_combo_pearson.append(float(pearsonr(pred, real)[0]))

    print(f'{genes[0]}+{genes[1]}: MSE={mse:.4f}, Pearson delta={per_combo_pearson[-1]:.4f}')

per_combo_mse = np.array(per_combo_mse)
per_combo_pearson = np.array(per_combo_pearson)

print(f'\nPer-combo MSE: mean={per_combo_mse.mean():.4f}, median={np.median(per_combo_mse):.4f}')
print(f'Per-combo Pearson: mean={per_combo_pearson.mean():.4f}')
print(f'\nSample-level MSE: {np.mean(sample_mses):.4f}')
print(f'Sample-level Pearson: {np.mean(sample_correlations):.4f}')

## Additive Baseline

The simplest prediction for what happens when you perturb genes A and B together is to just add their individual effects. If knocking out A upregulates gene 3 by 0.5 and knocking out B upregulates gene 3 by 0.3, the additive model predicts an upregulation of 0.8. This is a strong baseline because many biological effects are approximately additive. If our model can't beat this, it hasn't learned anything about how genes interact beyond their individual effects.

**Building Additive Deltas**

For each combo, we look up the two constituent gene names from `combo_to_genes`, find their rows in `single_deltas`, and sum them. We calculate the additive delta as:
$$
\delta^{\text{add}}_g = \delta^{\text{single\_A}}_g + \delta^{\text{single\_B}}_g
$$

You'll be able to see side by side how the additive prediction compares to the real combo effect. For the synergistic combo (GENE_1+GENE_2), you'll see the real effect is roughly 1.5x the additive prediction. For the alleviating combo (GENE_1+GENE_3), the real effect is much weaker than the sum, meaning the genes partially cancel each other.

In [ ]:
gene_name_to_idx = {g: i for i, g in enumerate(single_gene_names)}

additive_deltas = {}
for key in combo_keys:
    genes = combo_to_genes[key]
    idx_a = gene_name_to_idx[genes[0]]
    idx_b = gene_name_to_idx[genes[1]]
    additive = single_deltas[idx_a] + single_deltas[idx_b]
    additive_deltas[key] = additive

    print(f'---- {genes[0]} + {genes[1]} ----')
    print(f'  {genes[0]} single: {single_deltas[idx_a]}')
    print(f'  {genes[1]} single: {single_deltas[idx_b]}')
    print(f'  additive sum:     {additive}')
    print(f'  real combo:       {mean_real_deltas[key]}')
    print()

### All Genes

Now we compare both predictions against reality across all genes. For each combo, we compute the MSE of the model prediction and the MSE of the additive prediction against the real combo delta:
$$
\text{MSE}_{\text{model}} = \frac{1}{G}\sum_{g=1}^{G}(\hat{\delta}_g - \delta_g)^2 \qquad \text{MSE}_{\text{add}} = \frac{1}{G}\sum_{g=1}^{G}(\delta^{\text{add}}_g - \delta_g)^2
$$

Because of how we staged the data, you'll see the model wins on the synergistic combo (it learned the amplification) and the roughly-additive combo (it fine-tunes beyond the sum). The additive baseline wins on the alleviating combo (the model over-predicts) and the suppressive combo (the model over-predicts GENE_3's contribution).

In [ ]:
per_key_results = {}
model_mses_list = []
additive_mses_list = []

for key in combo_keys:
    genes = combo_to_genes[key]
    real = mean_real_deltas[key]
    pred = mean_pred_deltas[key]
    additive = additive_deltas[key]

    model_mse = float(np.mean((pred - real) ** 2))
    additive_mse = float(np.mean((additive - real) ** 2))
    model_mses_list.append(model_mse)
    additive_mses_list.append(additive_mse)

    per_key_results[key] = {'genes': genes, 'model_mse': model_mse, 'additive_mse': additive_mse, 'additive_delta': additive}

    winner = 'MODEL' if model_mse < additive_mse else 'ADDITIVE'
    print(f'{genes[0]}+{genes[1]}: model_mse={model_mse:.4f}, additive_mse={additive_mse:.4f} -> {winner} wins')

print(f'\nMean model MSE: {np.mean(model_mses_list):.4f}')
print(f'Mean additive MSE: {np.mean(additive_mses_list):.4f}')

**Pearson Comparison**

We also compute Pearson correlation for model vs real and additive vs real. Pearson captures whether the *shape* of the delta profile is correct, even if the magnitude is off. You'll see the results track the MSE pattern: the model has high Pearson on the synergistic combo where it learned the interaction, but both model and additive struggle on the combos with strong non-additive effects.

In [ ]:
model_pearsons = []
additive_pearsons = []

for key in combo_keys:
    genes = combo_to_genes[key]
    real = mean_real_deltas[key]
    pred = mean_pred_deltas[key]
    additive = additive_deltas[key]

    m_r = float(pearsonr(pred, real)[0])
    a_r = float(pearsonr(additive, real)[0])
    model_pearsons.append(m_r)
    additive_pearsons.append(a_r)

    print(f'{genes[0]}+{genes[1]}: model_pearson={m_r:.4f}, additive_pearson={a_r:.4f}')

print(f'\nMean model Pearson: {np.mean(model_pearsons):.4f}')
print(f'Mean additive Pearson: {np.mean(additive_pearsons):.4f}')

**Model Beats Additive Rate**

The headline number for this sub-eval: what fraction of combos does the model produce a lower MSE than the additive baseline? We calculate it as:
$$
\text{beat\_rate} = \frac{1}{C}\sum_{c=1}^{C}\mathbf{1}[\text{MSE}^{\text{model}}_c < \text{MSE}^{\text{add}}_c]
$$

Based on our staging, we expect a rate of 0.50, since the model wins on two combos (synergistic and roughly-additive) and loses on two (alleviating and suppressive).

In [ ]:
beat_rate = float(np.mean([m < a for m, a in zip(model_mses_list, additive_mses_list)]))
beat_rate

### Non-Additive Genes

The interesting biology is at the genes where the combo effect deviates most from the additive expectation. These are the genes where genetic interaction effects are strongest. For each combo, we identify the top-N genes with the largest *non-additive deviation*:
$$
\text{deviation}_g = |\delta^{\text{real}}_g - \delta^{\text{add}}_g|
$$

We then measure how well the model predicts specifically those genes using MSE and Pearson. If the model is capturing interaction effects, it should perform well on these genes. With 8 genes, we'll use the top 3 instead of the production top 20. You'll see that for the synergistic combo, the most non-additive genes are the ones where both GENE_1 and GENE_2 have their strongest effects (genes 0-3), which is where the amplification happens.

In [ ]:
TOP_N = 3
nonadd_model_mses = []
nonadd_pearsons = []

for key in combo_keys:
    genes = combo_to_genes[key]
    real = mean_real_deltas[key]
    pred = mean_pred_deltas[key]
    additive = additive_deltas[key]

    deviation = np.abs(real - additive)
    top_idx = np.argsort(deviation)[-TOP_N:]

    print(f'---- {genes[0]}+{genes[1]} ----')
    print(f'  deviation per gene: {np.round(deviation, 3)}')
    print(f'  top {TOP_N} gene indices: {top_idx}')
    print(f'  deviation at those: {deviation[top_idx]}')

    mse = float(np.mean((pred[top_idx] - real[top_idx]) ** 2))
    nonadd_model_mses.append(mse)
    nonadd_pearsons.append(float(pearsonr(pred[top_idx], real[top_idx])[0]))

    print(f'  model MSE on top-{TOP_N}: {mse:.4f}')
    print()

print(f'Non-additive gene analysis (top {TOP_N} genes per combo):')
print(f'  Mean MSE: {np.mean(nonadd_model_mses):.4f}')
print(f'  Mean Pearson: {np.mean(nonadd_pearsons):.4f}')

## Genetic Interaction Subtypes

Different gene pairs interact in different ways. The Norman dataset provides biological labels for these interaction types, classified by the GEARS project. By grouping our results by subtype, we can understand if the model is better at predicting certain types of interactions. We evaluate three metrics per subtype: mean model MSE, mean additive MSE (to see if one subtype is inherently harder), and interaction Pearson (which measures whether the model captures the non-additive component of the effect).

### Mapping Combos to Subtypes

We use `name_to_subtype` (built in data prep with both orderings of each gene pair) to find each combo's subtype. Not every combo will have a label. You'll see that GENE_2+GENE_4 has no known interaction type, so it's excluded from the per-subtype analysis.

In [ ]:
labeled_combos = {}
for key, kres in per_key_results.items():
    genes = kres['genes']
    combo_name = f'{genes[0]}_{genes[1]}'
    subtype = name_to_subtype.get(combo_name)
    if subtype:
        labeled_combos[key] = {'genes': genes, 'subtype': subtype, 'additive_delta': kres['additive_delta']}
        print(f'{genes[0]}+{genes[1]}: {subtype}')
    else:
        print(f'{genes[0]}+{genes[1]}: no label')

f'{len(labeled_combos)} of {len(combo_keys)} combos labeled'

### Interaction Delta

The *interaction effect* is what remains after subtracting the additive expectation:
$$
\text{interaction}^{\text{real}}_g = \delta^{\text{real}}_g - \delta^{\text{add}}_g \qquad \text{interaction}^{\text{pred}}_g = \delta^{\text{pred}}_g - \delta^{\text{add}}_g
$$

Pearson between these two vectors tells us whether the model has learned the direction and relative magnitude of the gene-gene interaction, separate from the baseline single-gene effects. A positive Pearson means the model captures the interaction pattern, even if the magnitudes are off. You'll see that for the synergistic combo, the interaction delta is large and positive (amplification), while for the alleviating combo it's large and negative (dampening).

In [ ]:
for key, info in labeled_combos.items():
    genes = info['genes']
    real = mean_real_deltas[key]
    pred = mean_pred_deltas[key]
    additive = info['additive_delta']

    interaction_real = real - additive
    interaction_pred = pred - additive

    r = float(pearsonr(interaction_pred, interaction_real)[0])

    print(f'---- {genes[0]}+{genes[1]} ({info["subtype"]}) ----')
    print(f'  interaction_real: {np.round(interaction_real, 3)}')
    print(f'  interaction_pred: {np.round(interaction_pred, 3)}')
    print(f'  interaction Pearson: {r:.4f}')
    print()

### Per-Subtype Aggregation

Now we group results by subtype. For each subtype, we compute mean model MSE, mean additive MSE, and mean interaction Pearson. Because of how we staged the data, you'll see that the synergistic combo shows the strongest model performance (it learned the amplification pattern), while the suppressive and alleviating combos are harder. The additive MSE column also tells us which subtypes deviate most from additivity: lower additive MSE means the real effect was closer to additive.

In [ ]:
by_subtype = defaultdict(lambda: {'model_mses': [], 'additive_mses': [], 'interaction_pearsons': []})

for key, info in labeled_combos.items():
    subtype = info['subtype']
    real = mean_real_deltas[key]
    pred = mean_pred_deltas[key]
    additive = info['additive_delta']

    by_subtype[subtype]['model_mses'].append(float(np.mean((pred - real) ** 2)))
    by_subtype[subtype]['additive_mses'].append(float(np.mean((additive - real) ** 2)))

    interaction_real = real - additive
    interaction_pred = pred - additive
    by_subtype[subtype]['interaction_pearsons'].append(float(pearsonr(interaction_pred, interaction_real)[0]))

print(f'{"Subtype":<15} {"N":>3} {"Model MSE":>10} {"Add MSE":>10} {"Int Pearson":>12}')
print('-' * 55)
for subtype, vals in sorted(by_subtype.items()):
    n = len(vals['model_mses'])
    m_mse = np.mean(vals['model_mses'])
    a_mse = np.mean(vals['additive_mses'])
    i_pear = np.mean(vals['interaction_pearsons'])
    print(f'{subtype:<15} {n:>3} {m_mse:>10.4f} {a_mse:>10.4f} {i_pear:>12.4f}')

## Combination Perturbation Final Wrapup

We've now walked through how BioJEPA-AC's combination perturbation eval works: starting from the composite key format that represents multi-pert samples, through basic expression prediction quality, then comparing against the additive baseline across all genes and on the most non-additive genes, and finally breaking results down by genetic interaction subtype.

The additive baseline is deliberately a strong benchmark. Many biological effects are approximately additive, so beating this baseline requires the model to have learned something about how specific gene pairs interact beyond their individual effects. As our walkthrough shows, the model captures some interactions well (synergistic amplification in GENE_1+GENE_2) but struggles with others (the cancellation in GENE_1+GENE_3, the dominance in GENE_3+GENE_4). The interaction Pearson and GI subtype breakdown help us understand which types of interactions the model handles well and which need improvement.